In [ ]:
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.linear_model import LinearRegression

# =========================
# Data path
# =========================

paths = config.CA_PATHS

# =========================
# Output folder
# =========================

out_dir = config.ensure_dir(config.FEATURES)
os.makedirs(out_dir, exist_ok=True)

# =========================
# Save raster
# =========================

def save_raster(path, data, src, dtype):

    with rasterio.open(
        path,
        'w',
        driver='GTiff',
        height=data.shape[0],
        width=data.shape[1],
        count=1,
        dtype=dtype,
        crs=src.crs,
        transform=src.transform,
        nodata=np.nan if "float" in dtype else 0
    ) as dst:

        dst.write(data.astype(dtype), 1)

# =========================
# Data cleaning
# =========================

def clean_data(data, src):

    data = data.astype(np.float64)

    if src.nodata is not None:
        data[data == src.nodata] = np.nan

    data[data > 1e20] = np.nan
    data[data <= 0] = np.nan

    return data

# =========================
# C-A Piecewise fitting合
# =========================

from sklearn.linear_model import LinearRegression
import numpy as np

def find_two_breakpoints(logC, logA):

    best_rss = np.inf

    best_i = None
    best_j = None

    n = len(logC)

    for i in range(15, n-30):

        for j in range(i+15, n-15):

            x1 = logC[:i].reshape(-1,1)
            y1 = logA[:i]

            x2 = logC[i:j].reshape(-1,1)
            y2 = logA[i:j]

            x3 = logC[j:].reshape(-1,1)
            y3 = logA[j:]

            m1 = LinearRegression()
            m2 = LinearRegression()
            m3 = LinearRegression()

            m1.fit(x1,y1)
            m2.fit(x2,y2)
            m3.fit(x3,y3)

            rss1 = np.sum((y1-m1.predict(x1))**2)
            rss2 = np.sum((y2-m2.predict(x2))**2)
            rss3 = np.sum((y3-m3.predict(x3))**2)

            rss = rss1 + rss2 + rss3

            if rss < best_rss:

                best_rss = rss

                best_i = i
                best_j = j

    return best_i, best_j
# =========================
# Main Program
# =========================

thresholds = []

for name, path in paths.items():

    print(f"\nProcessing {name}")

    with rasterio.open(path) as src:

        data = src.read(1)

        data = clean_data(data, src)

        valid = ~np.isnan(data)

        values = data[valid]

        print("Number of valid pixels:", len(values))

        # =========================
        # C-A curve
        # =========================

        c_values = np.logspace(
            np.log10(values.min()),
            np.log10(values.max()),
            100
        )

        areas = []

        for c in c_values:

            area = np.sum(values >= c)

            if area < 1:
                area = 1

            areas.append(area)

        areas = np.array(areas)

        logC = np.log10(c_values)
        logA = np.log10(areas)

        # =========================
        # Automatically find split points
        # =========================

        idx1, idx2 = find_two_breakpoints(logC, logA)

        th1 = c_values[idx1]
        th2 = c_values[idx2]


        print("Weak abnormal threshold:", th1)
        print("Strong anomaly threshold:", th2)
        
        thresholds.append([
            name,
            float(th1),
            float(th2)
        ])
        # =========================
        # C-A abnormal chart
        # =========================

        anomaly = np.zeros_like(
            data,
            dtype=np.uint8
        )

        # Background
        anomaly[data < th1] = 0

        # mild anomaly
        anomaly[(data >= th1) &
        (data < th2)] = 1

        # Strong exception
        anomaly[data >= th2] = 2

        save_raster(
            os.path.join(out_dir, f"{name}_CA.tif"),
            anomaly,
            src,
            "uint8"
        )
        # =========================
        # RF input feature layer
        # log10 + MinMax Normalization
        # =========================

        rf_feature = np.full(
            data.shape,
            np.nan,
            dtype=np.float32
        )

        log_data = np.log10(values)

        vmin = log_data.min()
        vmax = log_data.max()

        norm = (log_data - vmin) / (vmax - vmin + 1e-8)

        rf_feature[valid] = norm

        save_raster(
            os.path.join(out_dir, f"{name}_RF.tif"),
            rf_feature,
            src,
            "float32"
        )

        # =========================
        # Save the C-A curve
        # =========================

        plt.figure(figsize=(7, 5))

        plt.plot(
            logC,
            logA,
            'k.-',
            linewidth=1
        )
        plt.axvline(
            logC[idx1],
            color='orange',
            linestyle='--',
            label=f'Weak={th1:.2f}'
        )

        plt.axvline(
            logC[idx2],
            color='red',
            linestyle='--',
            label=f'Strong={th2:.2f}'
        )
        
        plt.xlabel('log(C)')
        plt.ylabel('log(A)')
        plt.title(f'{name} C-A Multifractal')

        plt.legend()

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                out_dir,
                f"{name}_CA_curve.png"
            ),
            dpi=300
        )

        plt.close()

# =========================
# Output Threshold Table
# =========================

df = pd.DataFrame(
    thresholds,
    columns=[
        "Element",
        "Weak_Threshold",
        "Strong_Threshold"
    ]
)

print(df)

df.to_excel(
    os.path.join(out_dir, "CA_thresholds.xlsx"),
    index=False
)


print("\nAll completed!")
print(df)
print(f"\nSave results to：{out_dir}")


In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================
# Global Settings
# ==========================
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 16
plt.rcParams['axes.linewidth'] = 1.2

# ==========================
# Data path
# ==========================
paths = config.CA_PATHS

out_dir = config.FEATURES

# ==========================
# Threshold
# ==========================
threshold_df = pd.read_excel(
    os.path.join(out_dir, "CA_thresholds.xlsx")
)

# ==========================
# Data cleaning
# ==========================
def clean_data(data, src):

    data = data.astype(np.float64)

    if src.nodata is not None:
        data[data == src.nodata] = np.nan

    data[data > 1e20] = np.nan
    data[data <= 0] = np.nan

    return data

# ==========================
# Drawing
# ==========================
for name, path in paths.items():

    print(f"Draw {name}")

    th1 = threshold_df.loc[
        threshold_df["Element"] == name,
        "Weak_Threshold"
    ].values[0]

    th2 = threshold_df.loc[
        threshold_df["Element"] == name,
        "Strong_Threshold"
    ].values[0]

    with rasterio.open(path) as src:

        data = clean_data(src.read(1), src)

    values = data[~np.isnan(data)]

    c_values = np.logspace(
        np.log10(values.min()),
        np.log10(values.max()),
        100
    )

    areas = np.array([
        max(np.sum(values >= c), 1)
        for c in c_values
    ])

    logC = np.log10(c_values)
    logA = np.log10(areas)

    idx1 = np.argmin(np.abs(c_values - th1))
    idx2 = np.argmin(np.abs(c_values - th2))

    plt.figure(figsize=(8,6))

    # C-A curve
    plt.plot(
        logC,
        logA,
        'k.-',
        linewidth=2,
        markersize=6
    )

    # Double pivot point
    plt.scatter(
        logC[idx1],
        logA[idx1],
        color='orange',
        s=80,
        zorder=5
    )

    plt.scatter(
        logC[idx2],
        logA[idx2],
        color='red',
        s=80,
        zorder=5
    )

    # Threshold line
    plt.axvline(
        logC[idx1],
        color='orange',
        linestyle='--',
        linewidth=2,
        label=f'Weak = {th1:.2f}'
    )

    plt.axvline(
        logC[idx2],
        color='red',
        linestyle='--',
        linewidth=2,
        label=f'Strong = {th2:.2f}'
    )

    plt.xlabel("log(C)", fontsize=26)
    plt.ylabel("log(A)", fontsize=26)
    plt.title(f"{name} C-A Multifractal", fontsize=24)

    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)

    plt.tick_params(
        direction='in',
        length=6,
        width=1.2
    )

    plt.legend(
        fontsize=20,
        frameon=True,
        loc='best'
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(out_dir, f"{name}_CA_curve_paper.png"),
        dpi=600,
        bbox_inches='tight'
    )

    plt.close()

print("All done！")